In [ ]:
# !pip install -q --upgrade torch==2.5.1+cu124 torchvision==0.20.1+cu124 torchaudio==2.5.1+cu124 --index-url https://download.pytorch.org/whl/cu124

!pip install -q --upgrade requests==2.32.3 bitsandbytes transformers==4.48.3 accelerate==1.3.0 datasets peft trl==0.14.0 matplotlib fsspec wandb

In [ ]:
! uname -r

In [ ]:
! pip show torch bitsandbytes transformers accelerate datasets peft trl fsspec wandb | awk '$1=="Name:" || $1 == "Version:"'

In [1]:
# With much thanks to Islam S. for identifying that there was a missing import!
import os, re, math
from datetime import datetime
os.environ['HF_HOME'] = "data/huggingface"
os.environ['WANDB_DIR'] = "data/wandb"
os.environ['WANDB_ARTIFACT_DIR'] = "data/artiface"

import dotenv, yaml
dotenv.load_dotenv("configs/local.env")

In [ ]:
from google.colab import userdata

os.environ['HF_USER'] = userdata.get('HF_USER')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

In [ ]:
from huggingface_hub import login
import wandb
from datasets import load_dataset, Dataset, DatasetDict
import matplotlib.pyplot as plt

#login(os.environ['HF_TOKEN'], add_to_git_credential=True)

# Run name for saving the model in the hub
DATASET_NAME = "ed-donner/pricer-data"
PROJECT_NAME = "pricer"
RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_ID = f"{os.environ['HF_USER']}/{PROJECT_RUN_NAME}"

# Configure Weights & Biases to record against our project
wandb.init(project=PROJECT_NAME, name=RUN_NAME)
# wandb.login(key=os.environ['WANDB_API_KEY'], force=True)

os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "checkpoint" # if LOG_TO_WANDB else "end"
os.environ["WANDB_WATCH"] = "gradients"

In [3]:
from tqdm import tqdm
import torch, transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

In [4]:
# Train parameters
BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"
MAX_SEQUENCE_LENGTH = 182

# Hyperparameters for QLoRA
LORA_R = 32
LORA_ALPHA = 64
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]
LORA_DROPOUT = 0.1

# Hyperparameters for Training
EPOCHS = 1 # you can do more epochs if you wish, but only 1 is needed - more is probably overkill
BATCH_SIZE = 4 # on an A100 box this can go up to 16
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 1e-4
LR_SCHEDULER_TYPE = 'cosine'
WARMUP_RATIO = 0.03
OPTIMIZER = "paged_adamw_32bit"

# Admin config - note that SAVE_STEPS is how often it will upload to the hub
# I've changed this from 5000 to 2000 so that you get more frequent saves
STEPS = 50
SAVE_STEPS = 2000

In [5]:
dataset = load_dataset(DATASET_NAME)
train, test = dataset['train'], dataset['test']

In [6]:
ls data/huggingface/datasets

data_huggingface_datasets_ed-donner___pricer-data_default_0.0.0_f38df5f756eafe7e331348b26a823e4d706f4a08.lock
ed-donner___pricer-data/


In [7]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
     bnb_4bit_compute_dtype=torch.bfloat16,

    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
# quant_config = BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=torch.bfloat16)

In [8]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:.1f} GB")

print("named_modules:", base_model.named_modules())
for name, module in base_model.named_modules():
    if any(x in name for x in ["proj", "fc"]):  # 你也可以 print 所有 name 先观察
        print(name)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Memory footprint: 4.0 GB


In [11]:
# First, specify the configuration parameters for LoRA
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

# Next, specify the general configuration parameters for training
train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    eval_strategy="no",
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb", # if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    #max_seq_length=MAX_SEQUENCE_LENGTH,
    dataset_text_field="text",
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_ID,
    hub_private_repo=True,
)

# from trl import DataCollatorForCompletionOnlyLM
# collator = DataCollatorForCompletionOnlyLM("Price is $", tokenizer=tokenizer)

Converting train dataset to ChatML:   0%|          | 0/400000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/400000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/400000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/400000 [00:00<?, ? examples/s]

In [ ]:
# And now, the Supervised Fine Tuning Trainer will carry out the fine-tuning
# Given these 2 sets of configuration parameters
# The latest version of trl is showing a warning about labels - please ignore this warning
# But let me know if you don't see good training results (loss coming down).

fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    peft_config=lora_parameters,
    args=train_parameters,
    # data_collator=collator,
)

In [ ]:
# Fine-tune!
fine_tuning.train()

In [ ]:
# Push our fine-tuned model to Hugging Face
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}")

wandb.finish()